In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=== RANDOM FOREST v3 - COLDTRACK (Versión Realista) ===\n")

# Cargar datos
X = pd.read_csv("X_features.csv")
y = pd.read_csv("y_target.csv").squeeze()

# ====================== LIMPIEZA AGRESIVA DE FEATURES (Anti-Leakage) ======================
features_to_remove = [
    'high_temp', 
    'critical_vibration', 
    'compressor_overwork', 
    'long_door_open',
    'ConsumoElectrico',           # ← quitamos consumo directo
    'Vibration',                  # ← quitamos vibración directa
    'inTemp_std_5min',
    'Vibration_std_5min'
]

X_clean = X.drop(columns=features_to_remove, errors='ignore')

print(f"Features después de limpieza estricta: {X_clean.shape[1]} columnas")
print("Columnas restantes:", X_clean.columns.tolist())

# ====================== ENTRENAMIENTO ======================
tscv = TimeSeriesSplit(n_splits=5)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,                    # Reducimos mucho la profundidad
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Usamos el último split temporal
train_index, test_index = list(tscv.split(X_clean))[-1]
X_train, X_test = X_clean.iloc[train_index], X_clean.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}\n")

model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# ====================== RESULTADOS ======================
print("="*65)
print("RESULTADOS DEL MODELO v3 - REALISTA")
print("="*65)
print(f"Accuracy     : {y_pred.mean():.4f}")
print(f"ROC AUC      : {roc_auc_score(y_test, y_pred_proba):.4f}\n")

print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Mantenimiento (1)']))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

# Importancia de features
importancia = pd.DataFrame({
    'Feature': X_clean.columns,
    'Importancia': model.feature_importances_
}).sort_values('Importancia', ascending=False)

print("\nTop 10 Features más importantes:")
print(importancia.head(10))

# Guardar modelo v3
joblib.dump(model, "coldtrack_rf_model_v3.pkl")
joblib.dump(X_clean.columns.tolist(), "feature_columns_v3.pkl")

print("\n✅ Modelo v3 guardado como: coldtrack_rf_model_v3.pkl")

=== RANDOM FOREST v3 - COLDTRACK (Versión Realista) ===

Features después de limpieza estricta: 17 columnas
Columnas restantes: ['inTemp', 'InHumid', 'outTemp', 'outHumid', 'temp_diff_setpoint', 'inTemp_ma_5min', 'inTemp_ma_15min', 'inTemp_ma_30min', 'inTemp_std_15min', 'Vibration_ma_5min', 'Vibration_ma_15min', 'ConsumoElectrico_ma_5min', 'door_open_last_hour', 'inTemp_trend_30min', 'hour', 'day_of_week', 'is_weekend']
Train: 259,200 | Test: 51,840

RESULTADOS DEL MODELO v3 - REALISTA
Accuracy     : 0.0858
ROC AUC      : 0.9994

Reporte de Clasificación:
                   precision    recall  f1-score   support

       Normal (0)       1.00      0.99      1.00     47722
Mantenimiento (1)       0.93      1.00      0.96      4118

         accuracy                           0.99     51840
        macro avg       0.96      1.00      0.98     51840
     weighted avg       0.99      0.99      0.99     51840


Matriz de Confusión:
[[47390   332]
 [    0  4118]]

Top 10 Features más importa